# Bank Loan Portfolio — Data Quality Audit

This diagnostic notebook tests whether the supplied extract is safe for portfolio-risk analysis. Warnings are retained as evidence; rows are never silently deleted.


## Summary

- Core grain, status, amount, rate, and issue-date guardrails are expected to pass.
- Cross-field payment and credit-pull dates are not governed and must not support timing or maturity claims.
- `emp_title` missingness and high-income outliers are retained and documented.



## Context & Methods

### Key Assumptions
- Source grain is one row per loan ID.
- `issue_date` is the only date used for origination trends.
- Payment/credit-pull chronology remains a source-validation issue, not a cleaning target.


In [1]:
from pathlib import Path
import sys
import pandas as pd

from IPython.display import display

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / 'data' / 'financial_loan.csv').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Cannot find data/financial_loan.csv')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_quality import run_full_audit
DATA_PATH = PROJECT_ROOT / 'data' / 'financial_loan.csv'
raw_loans = pd.read_csv(DATA_PATH)
print(f'Loaded {len(raw_loans):,} rows × {raw_loans.shape[1]} columns')


Loaded 38,576 rows × 24 columns


## Data


In [2]:
display(raw_loans.head(10))
display(raw_loans.dtypes.rename('dtype').to_frame())


,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,sub_grade,term,verification_status,annual_income,dti,installment,int_rate,loan_amount,total_acc,total_payment
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,11-02-2021,13-09-2021,13-04-2021,...,C4,60 months,Source Verified,30000.0,0.0100,59.83,0.1527,2500,4,1009
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,01-01-2021,14-12-2021,15-01-2021,...,E1,36 months,Source Verified,48000.0,0.0535,109.43,0.1864,3000,4,3939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,05-01-2021,12-12-2021,09-01-2021,...,C5,36 months,Not Verified,50000.0,0.2088,421.65,0.1596,12000,11,3522
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,25-02-2021,12-12-2021,12-03-2021,...,B2,60 months,Source Verified,42000.0,0.0540,97.06,0.1065,4500,9,4911
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,01-01-2021,14-12-2021,15-01-2021,...,A1,36 months,Verified,83000.0,0.0231,106.53,0.0603,3500,28,3835
5,1062608,CA,INDIVIDUAL,3 years,Studio 94 Corp,C,RENT,17-07-2021,16-03-2021,12-08-2021,...,C3,36 months,Not Verified,28000.0,0.1260,275.96,0.1465,8000,11,8637
6,1067441,TX,INDIVIDUAL,10+ years,American Airlines,C,MORTGAGE,19-11-2021,14-06-2021,13-12-2021,...,C2,36 months,Verified,94800.0,0.2453,205.86,0.1427,6000,30,7218
7,1066424,PA,INDIVIDUAL,10+ years,SCI Mahanoy,A,OWN,11-06-2021,14-07-2021,14-07-2021,...,A4,36 months,Source Verified,59000.0,0.0665,172.10,0.0790,5500,23,6172
8,1065254,FL,INDIVIDUAL,10+ years,Tech Data Corp,A,MORTGAGE,02-09-2021,15-06-2021,12-10-2021,...,A5,36 months,Verified,116400.0,0.0394,762.08,0.0890,24000,31,8650
9,1064589,MI,INDIVIDUAL,10+ years,teltow contracting,B,MORTGAGE,09-02-2021,16-03-2021,16-03-2021,...,B5,60 months,Not Verified,36000.0,0.1773,93.21,0.1269,4125,21,5551


,dtype
id,int64
address_state,str
application_type,str
emp_length,str
emp_title,str
grade,str
home_ownership,str
issue_date,str
last_credit_pull_date,str
last_payment_date,str


## Results

### 1. Run the governed audit


In [3]:
audit = run_full_audit(raw_loans)
display(audit['dq_summary'])
print(f"PASS={audit['n_pass']} | WARN={audit['n_warn']} | FAIL={audit['n_fail']}")


2026-09-03 20:53:04 | INFO     | ============================================================
2026-09-03 20:53:04 | INFO     | DATA QUALITY AUDIT -- Bank Loan Portfolio
2026-09-03 20:53:04 | INFO     | ============================================================
2026-09-03 20:53:04 | INFO     | [PASS] Row count — 38,576 rows, 24 columns (governed snapshot: 38,576)
2026-09-03 20:53:04 | INFO     | [PASS] Required columns — All required columns present
2026-09-03 20:53:04 | INFO     | [PASS] Duplicate rows — No fully-duplicated rows
2026-09-03 20:53:04 | INFO     | [PASS] Duplicate loan IDs — All loan IDs unique (guardrail satisfied)
2026-09-03 20:53:04 | WARNING  | [WARN] Missing: emp_title — 1,438 missing (3.73%)
2026-09-03 20:53:04 | INFO     | [PASS] loan_amount > 0 — 0 violations (0.00%)
2026-09-03 20:53:04 | INFO     | [PASS] installment > 0 — 0 violations (0.00%)
2026-09-03 20:53:04 | INFO     | [PASS] annual_income > 0 — 0 violations (0.00%)
2026-09-03 20:53:04 | INFO     | [PASS

,check,status,detail,value
0,Row count,PASS,"38,576 rows, 24 columns (governed snapshot: 38...","{'rows': 38576, 'cols': 24}"
1,Required columns,PASS,All required columns present,None
2,Duplicate rows,PASS,No fully-duplicated rows,None
3,Duplicate loan IDs,PASS,All loan IDs unique (guardrail satisfied),None
4,Missing: emp_title,WARN,"1,438 missing (3.73%)",None
5,loan_amount > 0,PASS,0 violations (0.00%),0
6,installment > 0,PASS,0 violations (0.00%),0
7,annual_income > 0,PASS,0 violations (0.00%),0
8,total_payment > 0,PASS,0 violations (0.00%),0
9,"int_rate in (0, 1]",PASS,0 violations (0.00%),0


PASS=23 | WARN=9 | FAIL=0


### 2. Review missingness and normalization


In [4]:
display(audit['missing_values'])
display(audit['string_normalization'])


,column,missing_count,missing_pct,strategy
0,emp_title,1438,3.727706,Fill 'Unknown' for grouping display; retain `e...


,column,affected_rows,affected_rate
0,term,38576,1.0
1,grade,0,0.0
2,purpose,0,0.0
3,emp_length,0,0.0
4,home_ownership,0,0.0


### 3. Review temporal contradictions


In [5]:
display(audit['temporal_consistency'].style.format({'affected_rate': '{:.2%}'}))


,rule,affected_rows,affected_rate,risk
0,last_payment_before_issue,15453,40.06%,Payment timing and vintage analysis
1,last_credit_pull_before_issue,20182,52.32%,Credit-pull timing and feature chronology
2,next_payment_before_last_payment,0,0.00%,Payment schedule chronology
3,resolved_with_next_payment_date,37478,97.15%,Status/date consistency


### 4. Review outliers and status coverage


In [6]:
display(audit['outlier_summary'])
display(audit['status_distribution'])


,column,p01,p25,p50 (median),p75,p99,"IQR fence [lower, upper]",n_below_fence,n_above_fence,classification,recommendation
0,annual_income,15000.0000,41500.0000,60000.0000,83200.5000,235000.0000,"[-21,050.75, 145,751.25]",0,1824,needs_business_validation,"1,824 records exceed upper fence ($145,751). R..."
1,loan_amount,1200.0000,5500.0000,10000.0000,15000.0000,35000.0000,"[-8,750.00, 29,250.00]",0,1208,valid_extreme,Large loan amounts are product-design valid; r...
2,int_rate,0.0542,0.0932,0.1186,0.1459,0.2099,"[0.01, 0.22]",0,78,valid_extreme,Interest-rate extremes plausible for sub-prime...
3,dti,0.0036,0.0821,0.1342,0.1859,0.2666,"[-0.07, 0.34]",0,0,valid_extreme,DTI distribution within expected range.


,loan_status,count,pct_of_portfolio
0,Fully Paid,32145,83.33
1,Charged Off,5333,13.82
2,Current,1098,2.85


## Takeaways


In [7]:
blocking = audit['n_fail'] > 0
print('BLOCK KPI PUBLICATION' if blocking else 'CORE KPI GUARDRAILS PASS')
print('Temporal payment/credit-pull fields remain quarantined until source semantics are verified.')
print('No source rows were deleted or silently imputed.')


CORE KPI GUARDRAILS PASS
Temporal payment/credit-pull fields remain quarantined until source semantics are verified.
No source rows were deleted or silently imputed.
